# Module 8: The Mega Capstone Project (40+ Operations)

Welcome to the final boss. This notebook is a **Complete Revision** of Modules 1 through 7. 

### The Scenario
You are the Lead Data Scientist at a massive Global Retailer. The data engineering team has handed you a raw database dump consisting of THREE interconnected tables: `Customers`, `Products`, and `Sales`.

**The data is a nightmare.** It has missing values, memory-hogging types, string formatting errors, bad datetimes, and broken foreign keys. 

### Instructions:
- There are **7 Phases** below containing dozens of operations.
- You must clean the tables individually, merge them perfectly, engineer features, and visualize the results.
- **NO ANSWERS ARE PROVIDED.** You have a blank code cell under each question. It's time to prove your skills.

## Phase 0: Generate the Relational Database

Run the cell below. It generates three massive CSV files (`customers.csv`, `products.csv`, `sales.csv`) on your machine and loads them.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)
n_sales = 50000
n_cust = 5000
n_prod = 100

# 1. CUSTOMERS TABLE
customers = pd.DataFrame({
    'cust_id': range(1, n_cust + 1),
    'Name': [f"User_{i}" for i in range(1, n_cust + 1)],
    'Country': np.random.choice([' USA ', 'uk', ' INDIA', 'Germany', 'Australia ', None], n_cust), # Messy strings, missing data
    'Sign_Up': [(datetime(2020, 1, 1) + timedelta(days=np.random.randint(0, 1000))).strftime('%Y-%m-%d') for _ in range(n_cust)]
})
# Inject duplicates
customers = pd.concat([customers, customers.sample(200)]).sample(frac=1).reset_index(drop=True)
customers.to_csv('customers.csv', index=False)

# 2. PRODUCTS TABLE
products = pd.DataFrame({
    'prod_id': [f"P-{str(i).zfill(3)}" for i in range(1, n_prod + 1)],
    'Category': np.random.choice(['Electronics', 'Clothing', 'Home', 'Toys'], n_prod),
    'Base_Price': [f"${np.random.uniform(10, 1000):.2f}" for _ in range(n_prod)] # Strings with $
})
products.to_csv('products.csv', index=False)

# 3. SALES TABLE
sales = pd.DataFrame({
    'txn_id': range(100000, 100000 + n_sales),
    'cust_id': np.random.randint(1, n_cust + 500, n_sales), # 500 invalid customer IDs that don't exist!
    'prod_id': [f"P-{str(np.random.randint(1, n_prod + 1)).zfill(3)}" for _ in range(n_sales)],
    'txn_date': [(datetime(2023, 1, 1) + timedelta(days=np.random.randint(0, 365), hours=np.random.randint(0, 23))).strftime('%d/%m/%Y %H:%M:%S') for _ in range(n_sales)],
    'qty': np.random.choice([1, 2, 3, 4, 5, -1, 999], n_sales), # Invalid negative and huge quantities
    'discount_pct': np.random.uniform(0.0, 0.5, n_sales)
})
# Add some NaNs to qty
sales.loc[np.random.choice(sales.index, 500), 'qty'] = np.nan
sales.to_csv('sales.csv', index=False)

print("Database generated: customers.csv, products.csv, sales.csv")

Database generated: customers.csv, products.csv, sales.csv


## Phase 1: Memory Optimization & Basic Cleaning (Customers Table)

1. Load `customers.csv`.
2. **Drop exact duplicate rows.**
3. The `Country` column has uppercase, lowercase, and trailing spaces. Clean it so it is purely Title Case with no spaces (e.g., `'Usa'`, `'Uk'`, `'India'`).
4. Fill missing `Country` values with `'Unknown'`.
5. Convert `Country` to a `category` data type to save memory.
6. Convert `cust_id` to an `int32`.
7. Print `customers.info(memory_usage='deep')` to prove it is optimized.

In [ ]:
# YOUR CODE HERE


## Phase 2: Regex & String Processing (Products Table)

1. Load `products.csv`.
2. The `Base_Price` column has `$` symbols and is treated as a string object. 
3. Use `.str.replace()` (with `regex=False`) to remove the `$`, convert the column to `float32`, and rename the column to `Price_USD`.
4. Check for any missing values. (There shouldn't be any, but prove it with code).

In [ ]:
# YOUR CODE HERE


## Phase 3: Datetime Wizardry & Anomalies (Sales Table)

1. Load `sales.csv`.
2. Convert `txn_date` to a proper Pandas datetime object. The format is exactly `DD/MM/YYYY HH:MM:SS`.
3. Extract three new columns: `Month` (1-12), `Day_Name` (e.g., Monday, Tuesday), and `Hour` (0-23).
4. Anomaly detection: The `qty` column has negative numbers (`-1`), massive outliers (`999`), and `NaN`s. 
   - Replace any quantity `< 1` or `> 10` with `np.nan` using `.where()` or boolean masking.
   - Fill all `NaN` quantities with `1.0`.
   - Convert `qty` to `int16`.

In [ ]:
# YOUR CODE HERE


## Phase 4: The Great Merge (Relational Joins)

You now have three clean DataFrames: `customers`, `products`, and `sales`.

1. Merge `sales` and `products` together on `prod_id`. Use a Left Join. Call this `df_merged`.
2. Merge `customers` into `df_merged` on `cust_id`. 
   - **Wait!** Remember that the Sales table had 500 fake `cust_id`s that don't exist in the Customers table? 
   - Use an **Inner Join** for this step to intentionally drop those fraudulent transactions.
3. Verify your final `df_merged` has no NaNs in the `Country` column.

In [ ]:
# YOUR CODE HERE


## Phase 5: Advanced Feature Engineering & Grouping

1. Create a `Final_Revenue` column: `(Price_USD * qty) * (1 - discount_pct)`.
2. **Window Function:** Sort the dataframe by `txn_date`. Group by `cust_id` and use `.cumsum()` on the `Final_Revenue` column to create a `Customer_Lifetime_Value` column. (This shows how much total money the customer has spent up to that specific transaction).
3. Use Named Aggregation (`.agg()`) to create a summary dataframe grouped by `Month` and `Category` calculating:
   - `Total_Sales` (sum of Final_Revenue)
   - `Avg_Discount` (mean of discount_pct)
   - `Transaction_Count` (count of txn_id)

In [ ]:
# YOUR CODE HERE


## Phase 6: Method Chaining Pipeline

Take the raw `sales.csv` file from disk again. Write ONE single method chain (using parentheses `()`) that:
1. Reads the CSV
2. Assigns `qty` to be 1 where `qty < 1` or `qty > 10`
3. Drops `discount_pct` column
4. Groups by `prod_id` and sums the `qty`
5. Sorts the output descending to find the top 5 most purchased product IDs.

In [ ]:
# YOUR CODE HERE


## Phase 7: Explanatory Visualization

Time to present to the CEO. Using your clean `df_merged` from Phase 5:

1. **Time Series:** Resample the data to a Weekly frequency (`'W'`) based on `txn_date`, sum the `Final_Revenue`, and plot a Line Chart of Weekly Sales. Use `figsize=(12,5)` and give it a title.
2. **Categorical:** Create a Pivot Table showing the sum of `Final_Revenue` where the index is `Country` and the columns are `Category`. 
3. **Heatmap:** Pass that Pivot Table into `sns.heatmap(annot=False, cmap='Blues')` to instantly show the CEO which country buys which category the most.
4. **Density:** Use a Seaborn `sns.violinplot()` with x='Category' and y='Final_Revenue' to show the price distributions of transactions in each category.

In [ ]:
# YOUR CODE HERE


## 🏆 MEGA CAPSTONE COMPLETE!

If you successfully answered these questions without having to look up the solutions, you are officially a Pandas Professional. 

You have proven you can handle massive relational data, engineer features, optimize memory, write pipelines, and build analytical charts. 

**You are ready.**